# 03_random_forest.ipynb — Random Forest Modeling & Forecasting

**UAS ADW** — Perbandingan SARIMA vs Tree-Based ML (Random Forest & XGBoost)

Tujuan: Feature engineering, training Random Forest, rolling origin forecasting (h=1,7,30), evaluasi metrik.

In [ ]:
# ==============================================================================
# SETUP & IMPORTS
# ==============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import joblib

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
OUTPUT_DIR = os.path.join(DATA_DIR, "rf_output")
FIGURES_DIR = os.path.join(DATA_DIR, "figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")

In [ ]:
# ==============================================================================
# LOAD DATA
# ==============================================================================
df = pd.read_csv(os.path.join(DATA_DIR, "seri_harian.csv"), parse_dates=["tanggal"])
df.set_index("tanggal", inplace=True)
df = df.asfreq("D")

komoditas = ["cabe_rawit", "bawang_merah", "beras_premium"]
print(f"Data: {len(df)} hari | Komoditas: {komoditas}")
print(f"Range: {df.index.min()} s/d {df.index.max()}")
print(df.head())
print(df.tail())

In [ ]:
# ==============================================================================
# FEATURE ENGINEERING
# ==============================================================================

def create_features(series, lags=None, windows=None):
    """
    Membuat features untuk time series forecasting.
    - Lags: t-1, t-7, t-30 dll
    - Rolling statistics: mean, std, min, max
    - Calendar features: dayofweek, month, quarter, is_weekend
    """
    if lags is None:
        lags = [1, 2, 3, 7, 14, 30]
    if windows is None:
        windows = [7, 14, 30]
    
    df_feat = pd.DataFrame(index=series.index)
    df_feat["target"] = series.values
    
    # Lag features
    for lag in lags:
        df_feat[f"lag_{lag}"] = series.shift(lag)
    
    # Rolling statistics
    for w in windows:
        df_feat[f"roll_mean_{w}"] = series.shift(1).rolling(w).mean()
        df_feat[f"roll_std_{w}"] = series.shift(1).rolling(w).std()
        df_feat[f"roll_min_{w}"] = series.shift(1).rolling(w).min()
        df_feat[f"roll_max_{w}"] = series.shift(1).rolling(w).max()
    
    # Calendar features
    df_feat["dayofweek"] = df_feat.index.dayofweek
    df_feat["month"] = df_feat.index.month
    df_feat["quarter"] = df_feat.index.quarter
    df_feat["dayofyear"] = df_feat.index.dayofyear
    df_feat["is_weekend"] = (df_feat.index.dayofweek >= 5).astype(int)
    
    # Drop NaN dari shifting/rolling
    df_feat = df_feat.dropna()
    
    return df_feat

# Test feature engineering untuk satu komoditas
test_feat = create_features(df["cabe_rawit"])
print(f"Feature shape: {test_feat.shape}")
print(f"Features: {list(test_feat.columns)}")
print(test_feat.head())

In [ ]:
# ==============================================================================
# DEFINE ROLLING ORIGIN FOLDS (Same as SARIMA)
# ==============================================================================

folds = [
    ("Fold1", df.index[(df.index >= "2023-01-01") & (df.index <= "2024-12-31")],
     df.index[(df.index >= "2025-01-01") & (df.index <= "2025-06-30")]),
    ("Fold2", df.index[(df.index >= "2023-01-01") & (df.index <= "2025-06-30")],
     df.index[(df.index >= "2025-07-01") & (df.index <= "2025-12-31")]),
    ("Fold3", df.index[(df.index >= "2023-01-01") & (df.index <= "2025-12-31")],
     df.index[(df.index >= "2026-01-01") & (df.index <= "2026-08-31")]),
]

horizons = [1, 7, 30]
max_h = max(horizons)

for fold_name, train_idx, test_idx in folds:
    print(f"{fold_name}: Train {train_idx[0].date()} to {train_idx[-1].date()} | Test {test_idx[0].date()} to {test_idx[-1].date()} | n_test={len(test_idx)}")

In [ ]:
# ==============================================================================
# ROLLING ORIGIN FORECASTING WITH RANDOM FOREST
# ==============================================================================

from joblib import Parallel, delayed
import time

def rf_forecast_one_fold(series, fold_name, train_idx, test_idx, horizons, max_h, n_estimators=200):
    """
    Rolling origin forecast untuk Random Forest.
    Model di-retrain di setiap origin (expanding window).
    """
    results = {h: {"y_true": [], "y_pred": [], "dates": []} for h in horizons}
    n_origin = len(test_idx) - max_h + 1
    
    for i in range(n_origin):
        origin = test_idx[i]
        # Data training up to origin
        train_series = series.loc[:origin]
        
        # Create features untuk training data
        train_feat = create_features(train_series)
        if len(train_feat) < 50:  # butuh data minimal
            continue
        
        X_train = train_feat.drop("target", axis=1)
        y_train = train_feat["target"]
        
        # Train RF
        rf = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        
        # Predict untuk tiap horizon
        for h in horizons:
            target_date = test_idx[i + h - 1]
            
            # Buat features untuk target_date (menggunakan history up to origin)
            # Kita perlu series up to target_date-1 untuk lag/rolling
            hist_series = series.loc[:target_date].iloc[:-1]  # exclude target
            if len(hist_series) < max([1,2,3,7,14,30]):
                continue
            
            # Buat single row features
            feat_row = {}
            # Lags
            for lag in [1, 2, 3, 7, 14, 30]:
                if len(hist_series) >= lag:
                    feat_row[f"lag_{lag}"] = hist_series.iloc[-lag]
                else:
                    feat_row[f"lag_{lag}"] = np.nan
            # Rolling
            for w in [7, 14, 30]:
                if len(hist_series) >= w:
                    recent = hist_series.iloc[-w:]
                    feat_row[f"roll_mean_{w}"] = recent.mean()
                    feat_row[f"roll_std_{w}"] = recent.std()
                    feat_row[f"roll_min_{w}"] = recent.min()
                    feat_row[f"roll_max_{w}"] = recent.max()
                else:
                    feat_row[f"roll_mean_{w}"] = np.nan
                    feat_row[f"roll_std_{w}"] = np.nan
                    feat_row[f"roll_min_{w}"] = np.nan
                    feat_row[f"roll_max_{w}"] = np.nan
            # Calendar
            feat_row["dayofweek"] = target_date.dayofweek
            feat_row["month"] = target_date.month
            feat_row["quarter"] = target_date.quarter
            feat_row["dayofyear"] = target_date.dayofyear
            feat_row["is_weekend"] = int(target_date.dayofweek >= 5)
            
            X_pred = pd.DataFrame([feat_row])
            
            try:
                y_pred = rf.predict(X_pred)[0]
                y_true = float(series.loc[target_date])
                
                results[h]["y_true"].append(y_true)
                results[h]["y_pred"].append(y_pred)
                results[h]["dates"].append(target_date)
            except Exception as e:
                continue
    
    return results

# Jalankan untuk semua komoditas dan fold
all_forecasts = []

for fold_name, train_idx, test_idx in folds:
    print(f"\n=== {fold_name} ===")
    
    for col in komoditas:
        print(f"  {col} ...", end=" ", flush=True)
        t0 = time.time()
        
        res = rf_forecast_one_fold(
            df[col], fold_name, train_idx, test_idx, horizons, max_h
        )
        
        for h in horizons:
            n = len(res[h]["y_true"])
            if n == 0:
                continue
            y_true = np.array(res[h]["y_true"])
            y_pred = np.array(res[h]["y_pred"])
            
            mae = mean_absolute_error(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
            
            for i in range(n):
                all_forecasts.append({
                    "fold": fold_name,
                    "komoditas": col,
                    "horizon": h,
                    "date": res[h]["dates"][i],
                    "y_true": y_true[i],
                    "y_pred": y_pred[i],
                    "abs_error": abs(y_true[i] - y_pred[i]),
                    "squared_error": (y_true[i] - y_pred[i])**2,
                    "mae_fold": mae,
                    "rmse_fold": rmse,
                    "mape_fold": mape,
                    "n_obs": n
                })
        
        print(f"done ({time.time()-t0:.1f}s)")

forecast_df = pd.DataFrame(all_forecasts)
forecast_df.to_csv(os.path.join(OUTPUT_DIR, "rf_forecast_all.csv"), index=False)
print(f"\nTotal forecast points: {len(forecast_df)}")
print(forecast_df.groupby(["fold", "komoditas", "horizon"]).size().unstack(fill_value=0))

In [ ]:
# ==============================================================================
# METRICS SUMMARY TABLE
# ==============================================================================

metrics_summary = forecast_df.groupby(["komoditas", "horizon"]).agg(
    MAE=("abs_error", "mean"),
    RMSE=("squared_error", lambda x: np.sqrt(x.mean())),
    MAPE=("abs_error", lambda x: np.mean(x / forecast_df.loc[x.index, "y_true"]) * 100),
    N=("abs_error", "count")
).reset_index()

print("=== RF Metrics (pooled across folds) ===")
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

metrics_summary.to_csv(os.path.join(OUTPUT_DIR, "rf_metrics_summary.csv"), index=False)

fold_metrics = forecast_df.groupby(["fold", "komoditas", "horizon"]).agg(
    MAE=("abs_error", "mean"),
    RMSE=("squared_error", lambda x: np.sqrt(x.mean())),
    MAPE=("abs_error", lambda x: np.mean(x / forecast_df.loc[x.index, "y_true"]) * 100),
    N=("abs_error", "count")
).reset_index()

fold_metrics.to_csv(os.path.join(OUTPUT_DIR, "rf_metrics_per_fold.csv"), index=False)
print("\n=== Per-Fold Metrics ===")
print(fold_metrics.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

In [ ]:
# ==============================================================================
# FORECAST VS ACTUAL PLOTS (h=1)
# ==============================================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for i, col in enumerate(komoditas):
    ax = axes[i]
    
    # Actual (full series)
    ax.plot(df.index, df[col], color='gray', linewidth=0.6, alpha=0.5, label='Actual (full)')
    
    # RF Forecasts h=1
    fc = forecast_df[(forecast_df['komoditas'] == col) & (forecast_df['horizon'] == 1)]
    
    for fold_name in ['Fold1', 'Fold2', 'Fold3']:
        fcf = fc[fc['fold'] == fold_name]
        if len(fcf):
            ax.plot(fcf['date'], fcf['y_pred'], 'o-', markersize=2, linewidth=1,
                   label=f'{fold_name} RF', alpha=0.8)
    
    ax.set_title(f"Random Forest Forecast vs Actual — {col} (h=1)", fontsize=12, fontweight='bold')
    ax.set_ylabel('Harga (Rp/kg)')
    ax.legend(loc='upper left', fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

axes[-1].set_xlabel('Tanggal')
plt.tight_layout()
fig_path = os.path.join(FIGURES_DIR, "06_rf_forecast_vs_actual.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Grafik tersimpan: {fig_path}")
plt.show()

In [ ]:
# ==============================================================================
# FEATURE IMPORTANCE (Full Data Model)
# ==============================================================================

importances_dict = {}

for col in komoditas:
    print(f"\n=== {col} ===")
    full_feat = create_features(df[col])
    X = full_feat.drop("target", axis=1)
    y = full_feat["target"]
    
    rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    
    imp = pd.DataFrame({
        "feature": X.columns,
        "importance": rf.feature_importances_
    }).sort_values("importance", ascending=False)
    
    importances_dict[col] = imp
    imp.to_csv(os.path.join(OUTPUT_DIR, f"rf_feature_importance_{col}.csv"), index=False)
    
    print(imp.head(15).to_string(index=False))
    
    # Plot top 15
    plt.figure(figsize=(8, 6))
    top15 = imp.head(15).iloc[::-1]
    plt.barh(top15["feature"], top15["importance"])
    plt.title(f"Feature Importance - {col} (Random Forest)")
    plt.xlabel("Importance")
    plt.tight_layout()
    fig_path = os.path.join(FIGURES_DIR, f"07_rf_feature_importance_{col}.png")
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ==============================================================================
# SAVE FINAL OUTPUTS
# ==============================================================================

print("=== Files Saved ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f}")

print("\n=== RF Metrics Summary ===")
print(metrics_summary.to_string(index=False, float_format=lambda x: f"{x:.2f}"))